# Similaridade entre documentos e versões

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/04_similaridade_documentos_e_versoes.ipynb)

O Notebook 03 produziu conjuntos de tokens, vetores de contagem e pesos TF-IDF.
Agora veremos que “semelhante” não é propriedade única do texto: cada medida
responde a uma relação diferente.

## 1. Escolher a medida pela pergunta

![Jaccard compara conjuntos, cosseno compara vetores, distância de edição compara sequências e a leitura próxima examina resultados divergentes.](imagens/04_escolha_metrica.svg)

| Pergunta | Representação | Medida inicial |
|---|---|---|
| Compartilham vocabulário? | conjunto de termos | Jaccard |
| Têm perfis de frequência parecidos? | vetor de contagens | cosseno |
| Compartilham termos distintivos? | vetor TF-IDF | cosseno |
| Quanto uma versão precisa mudar? | sequência de caracteres ou tokens | edição |

Autores, períodos e coleções podem ser comparados agregando documentos, desde
que autoria e pertencimento estejam documentados. Nesta base não há campo de
autoria; portanto, não inventaremos autores para cumprir um exemplo.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente local: nenhuma clonagem necessária.


In [2]:
import math
import re
import unicodedata
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display

dados = pd.read_csv("dados/documentos_comparacao.csv")
dados["texto"] = dados["texto_comparacao"]
stopwords = {"a", "e", "o", "de", "do", "da", "em", "nas", "também"}

def tokenizar(texto):
    base = unicodedata.normalize("NFKD", texto.lower())
    base = "".join(c for c in base if not unicodedata.combining(c))
    return [t for t in re.findall(r"[a-z]+", base) if t not in stopwords]

dados["tokens"] = dados["texto"].map(tokenizar)
matriz_contagens = pd.DataFrame(
    [Counter(t) for t in dados["tokens"]], index=dados["id_documento"]
).fillna(0).astype(int)
N = len(matriz_contagens)
df = matriz_contagens.gt(0).sum()
idf = ((N + 1) / (df + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)

Com as representações reconstruídas, começamos por ignorar frequências e
comparar somente presença e ausência de termos.

## 2. Similaridade de Jaccard

Para conjuntos de termos $A$ e $B$:

$$
J(A,B)=\frac{|A\cap B|}{|A\cup B|}.
$$

O resultado varia de 0 a 1. Repetir um termo não altera Jaccard nesta definição;
dois textos de tamanhos muito diferentes podem ser penalizados pela união ampla.

In [3]:
def jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    return len(a & b) / len(a | b) if a | b else 1.0

consulta = "D001"
tokens_consulta = dados.set_index("id_documento").loc[consulta, "tokens"]
similaridades_jaccard = dados.set_index("id_documento")["tokens"].map(
    lambda tokens: jaccard(tokens_consulta, tokens)
).drop(consulta).sort_values(ascending=False)
similaridades_jaccard.head(6).rename("Jaccard")

id_documento
D009    0.647059
D004    0.527778
D005    0.500000
D010    0.459459
D008    0.447368
D007    0.384615
Name: Jaccard, dtype: float64

Jaccard trata todos os termos presentes igualmente. Para preservar frequências
ou pesos, comparamos o ângulo entre vetores.

## 3. Similaridade de cosseno

$$
\cos(\mathbf{x},\mathbf{y})=
\frac{\mathbf{x}\cdot\mathbf{y}}{\|\mathbf{x}\|_2\|\mathbf{y}\|_2}.
$$

Em vetores não negativos, 1 indica mesma direção e 0 ausência de componentes
compartilhados. Cosseno reduz o efeito da magnitude total, mas continua
dependente do vocabulário, da ponderação e do pré-processamento.

In [4]:
def cosseno(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    denominador = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denominador) if denominador else 0.0

vetor_contagem = matriz_contagens.loc[consulta]
vetor_tfidf = matriz_tfidf.loc[consulta]
similaridades_cosseno_contagem = matriz_contagens.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_contagem, linha), axis=1
).sort_values(ascending=False)
similaridades_cosseno_tfidf = matriz_tfidf.drop(index=consulta).apply(
    lambda linha: cosseno(vetor_tfidf, linha), axis=1
).sort_values(ascending=False)
ranking = pd.concat([
    similaridades_jaccard.rename("Jaccard"),
    similaridades_cosseno_contagem.rename("cosseno contagens"),
    similaridades_cosseno_tfidf.rename("cosseno TF-IDF"),
], axis=1)
ranking.sort_values("cosseno TF-IDF", ascending=False).head(8).round(3)

,Jaccard,cosseno contagens,cosseno TF-IDF
id_documento,,,
D009,0.647,0.790,0.750
D004,0.528,0.691,0.674
D005,0.500,0.723,0.664
D010,0.459,0.632,0.610
D007,0.385,0.568,0.549
D008,0.447,0.593,0.524
D012,0.333,0.514,0.449
D002,0.385,0.504,0.431


Os três rankings podem divergir porque presença, repetição e raridade recebem
pesos diferentes. Essa divergência é material analítico, não um obstáculo a ser
escondido.

## 4. Identificar documentos semelhantes e inspecionar divergências

![Um documento consultado liga-se a vizinhos encontrados por Jaccard, cosseno de contagens e cosseno TF-IDF, incluindo um caso divergente.](imagens/04_vizinhos_documentais.svg)

Selecione pares que aparecem no topo de várias métricas e pares cuja posição
muda. Leia texto, metadados e extensão antes de chamá-los semelhantes.

In [5]:
ids_inspecao = list(dict.fromkeys(
    [consulta]
    + similaridades_jaccard.head(2).index.tolist()
    + similaridades_cosseno_tfidf.head(2).index.tolist()
))
pares_inspecao = dados.set_index("id_documento").loc[
    ids_inspecao, ["ano", "genero", "local", "tema", "texto"]
]
display(ranking.loc[[i for i in ids_inspecao if i != consulta]].round(3))
pares_inspecao

,Jaccard,cosseno contagens,cosseno TF-IDF
id_documento,,,
D009,0.647,0.790,0.750
D004,0.528,0.691,0.674


,ano,genero,local,tema,texto
id_documento,,,,,
D001,1890,editorial,Capital,educação,A escola noturna amplia a instrução pública e ...
D009,1898,carta,Capital,educação,A escola noturna amplia a instrução pública e ...
D004,1893,editorial,Interior,saúde,A cidade debate higiene água saúde e cuidado c...


Vetores são adequados para perfis lexicais. Para comparar versões quase iguais,
a ordem das unidades é crucial; passamos então à distância de edição.

## 5. Distância de edição e comparação de versões

A distância de Levenshtein é o menor número de inserções, exclusões e
substituições necessário para transformar uma sequência em outra:

$$
D_{i,j}=\min\begin{cases}
D_{i-1,j}+1 & \text{exclusão}\\
D_{i,j-1}+1 & \text{inserção}\\
D_{i-1,j-1}+\mathbf{1}(a_i\neq b_j) & \text{substituição}.
\end{cases}
$$

![Duas sequências curtas são alinhadas por operações de manter, substituir, inserir e excluir; a figura alerta que distância não interpreta o sentido.](imagens/04_distancia_edicao.svg)

A unidade pode ser caractere ou token. A distância bruta cresce com o tamanho;
uma versão normalizada divide pelo maior comprimento, com convenção declarada.

In [6]:
def levenshtein(seq_a, seq_b):
    anterior = list(range(len(seq_b) + 1))
    for i, a in enumerate(seq_a, 1):
        atual = [i]
        for j, b in enumerate(seq_b, 1):
            atual.append(min(
                anterior[j] + 1,
                atual[j-1] + 1,
                anterior[j-1] + (a != b),
            ))
        anterior = atual
    return anterior[-1]

versoes = pd.read_csv("dados/versoes_textuais.csv")
resultados_edicao = []
for id_documento, grupo in versoes.groupby("id_documento"):
    x, y = grupo.iloc[0], grupo.iloc[1]
    tokens_x, tokens_y = tokenizar(x["texto"]), tokenizar(y["texto"])
    distancia = levenshtein(tokens_x, tokens_y)
    resultados_edicao.append({
        "id_documento": id_documento,
        "versão A": x["tipo_versao"], "versão B": y["tipo_versao"],
        "distância em tokens": distancia,
        "distância normalizada": distancia / max(len(tokens_x), len(tokens_y)),
    })
tabela_edicao = pd.DataFrame(resultados_edicao)
tabela_edicao.round(3)

,id_documento,versão A,versão B,distância em tokens,distância normalizada
0,D001,transcrição A,transcrição B,0,0.000
1,D002,OCR bruto,revisada,0,0.000
2,D003,edição 1,edição 2,3,0.375


## Aplicação ao projeto Goodreads

As células seguintes aplicam as operações do notebook à base do projeto.
Especificação principal: **E2** (rótulo original sem os 1.274 livros
duvidosos): 4.638 livros de autoria feminina (F) e 4.542 de autoria masculina
(M). A ordem das diferenças é sempre **F − M**: valor negativo indica menos
reconhecimento para livros de autoras.

In [7]:
# Localiza a pasta do projeto Goodreads (localmente ou no Colab, se a pasta for enviada para /content)
import sys
from pathlib import Path

CANDIDATOS = [Path("../projeto_goodreads"), Path("projeto_goodreads"), Path("/content/projeto_goodreads")]
PROJETO = next((c.resolve() for c in CANDIDATOS if (c / "preparar_base.py").exists()), None)
if PROJETO is None:
    raise FileNotFoundError(
        "Pasta projeto_goodreads não encontrada. Coloque-a ao lado das pastas unidade_0X "
        "ou, no Colab, envie-a para /content."
    )
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))
print("Projeto localizado:", (PROJETO / "preparar_base.py").exists())
from preparar_base import carregar_base_analitica
import numpy as np
import pandas as pd
from IPython.display import display

base = carregar_base_analitica()
ESPEC = {"E1 original": "genero_E1_original",
         "E2 sem duvidosos (principal)": "genero_E2_sem_duvidosos",
         "E3 reclassificada": "genero_E3_reclassificado"}
GRUPO = "genero_E2_sem_duvidosos"
e2 = base[base[GRUPO].notna()].copy()
e2["muito_avaliado"] = e2["n_avaliacoes"] >= 10_000
F = e2[GRUPO].eq("feminino")
M = e2[GRUPO].eq("masculino")
print("Especificação principal E2:", int(F.sum()), "livros de autoria feminina e", int(M.sum()), "de autoria masculina")

import math
import re
import unicodedata
from collections import Counter
from graficos import mapa_calor

STOPWORDS_EN = {"the", "a", "an", "of", "and", "in", "to", "for", "on", "at", "with", "from",
                "by", "my", "your", "is", "or", "her", "his", "our", "their", "its", "be",
                "as", "into", "about", "this", "that", "what", "how", "who", "when", "you",
                "me", "i", "it", "not", "no", "all", "one", "s", "de", "la", "el", "le",
                "vol", "volume", "book", "series"}

def tokenizar_titulo(texto):
    texto = re.sub(r"\(.*?\)", " ", str(texto))          # retira marcador de série
    base_txt = unicodedata.normalize("NFKD", texto.casefold())
    base_txt = "".join(c for c in base_txt if not unicodedata.combining(c))
    return [t for t in re.findall(r"[^\W\d_]+", base_txt) if t not in STOPWORDS_EN]

fantasy = e2[e2["genero_1"].eq("Fantasy")].copy()
fantasy["tokens"] = fantasy["titulo"].map(tokenizar_titulo)
fantasy = fantasy[fantasy["tokens"].map(len) > 0]
contagens = [Counter(t) for t in fantasy["tokens"]]
matriz_contagens = pd.DataFrame(contagens, index=fantasy["id_livro"]).fillna(0).astype(int)
print("Documentos (títulos de Fantasy):", matriz_contagens.shape[0], "| vocabulário:", matriz_contagens.shape[1])
print("Proporção de células iguais a zero:", round(float((matriz_contagens == 0).mean().mean()), 4))
display(fantasy.groupby(GRUPO).size().rename("títulos"))


N = len(matriz_contagens)
df_termo = matriz_contagens.gt(0).sum(axis=0)
idf = ((N + 1) / (df_termo + 1)).map(math.log) + 1
matriz_tfidf = matriz_contagens.mul(idf, axis=1)

Projeto localizado: True
Especificação principal E2: 4638 livros de autoria feminina e 4542 de autoria masculina


Documentos (títulos de Fantasy): 1158 | vocabulário: 1524
Proporção de células iguais a zero: 0.9986


genero_E2_sem_duvidosos
feminino     675
masculino    483
Name: títulos, dtype: int64

**Jaccard e cosseno (contagens e TF-IDF)** para um título-consulta de Fantasy.

In [8]:
def jaccard(tokens_a, tokens_b):
    a, b = set(tokens_a), set(tokens_b)
    return len(a & b) / len(a | b) if a | b else 1.0

def cosseno(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    denominador = np.linalg.norm(x) * np.linalg.norm(y)
    return float(np.dot(x, y) / denominador) if denominador else 0.0

indice = fantasy.set_index("id_livro")
consulta = int(indice.index[indice["titulo"].eq("The Girl of Fire and Thorns")][0])
tokens_consulta = indice.loc[consulta, "tokens"]
sim_jaccard = indice["tokens"].map(lambda t: jaccard(tokens_consulta, t)).drop(consulta)
outros = matriz_contagens.drop(index=consulta)
sim_cos_contagem = pd.Series(outros.to_numpy() @ matriz_contagens.loc[consulta].to_numpy(), index=outros.index) / (
    np.linalg.norm(outros.to_numpy(), axis=1) * np.linalg.norm(matriz_contagens.loc[consulta].to_numpy()))
outros_tfidf = matriz_tfidf.drop(index=consulta)
sim_cos_tfidf = pd.Series(outros_tfidf.to_numpy() @ matriz_tfidf.loc[consulta].to_numpy(), index=outros_tfidf.index) / (
    np.linalg.norm(outros_tfidf.to_numpy(), axis=1) * np.linalg.norm(matriz_tfidf.loc[consulta].to_numpy()))
ranking = pd.concat([sim_jaccard.rename("Jaccard"), sim_cos_contagem.rename("cosseno contagens"),
                     sim_cos_tfidf.rename("cosseno TF-IDF")], axis=1)
ranking = ranking.join(indice[["titulo", "autor", GRUPO]])
print("Consulta:", indice.loc[consulta, "titulo"], "|", indice.loc[consulta, "autor"], "| tokens:", tokens_consulta)
ranking.sort_values(["cosseno TF-IDF", "Jaccard"], ascending=False).head(8).round(3)

Consulta: The Girl of Fire and Thorns | Rae Carson | tokens: ['girl', 'fire', 'thorns']


,Jaccard,cosseno contagens,cosseno TF-IDF,titulo,autor,genero_E2_sem_duvidosos
id_livro,,,,,,
25358353,0.333,0.577,0.518,Into the Fire (Under Fire #1),Kim Vandel,feminino
17452685,0.333,0.577,0.518,Into the Fire,Ashelyn Drake,feminino
12891107,0.250,0.408,0.513,King of Thorns,Mark Lawrence,masculino
15985373,0.250,0.408,0.450,Emperor of Thorns,Mark Lawrence,masculino
35105833,0.250,0.408,0.417,The Girl King,Mimi Yu,feminino
17454983,0.250,0.408,0.381,Alpha Girl,Kate Bloomfield,feminino
9481431,0.250,0.408,0.366,Demon Girl,Penelope Fletcher,feminino
12795973,0.250,0.408,0.358,Dust Girl,Sarah Zettel,feminino


In [9]:
# Divergência entre medidas: candidatos que mudam de posição
posicoes = ranking[["Jaccard", "cosseno contagens", "cosseno TF-IDF"]].rank(ascending=False, method="min")
candidatos = ranking.sort_values("cosseno TF-IDF", ascending=False).head(3).index.tolist()
candidatos += ranking.sort_values("Jaccard", ascending=False).head(3).index.tolist()
candidatos = list(dict.fromkeys(candidatos))
ranking.loc[candidatos].join(posicoes, rsuffix="_posicao").round(3)

,Jaccard,cosseno contagens,cosseno TF-IDF,titulo,autor,genero_E2_sem_duvidosos,Jaccard_posicao,cosseno contagens_posicao,cosseno TF-IDF_posicao
id_livro,,,,,,,,,
25358353,0.333,0.577,0.518,Into the Fire (Under Fire #1),Kim Vandel,feminino,1.0,1.0,1.0
17452685,0.333,0.577,0.518,Into the Fire,Ashelyn Drake,feminino,1.0,1.0,1.0
12891107,0.250,0.408,0.513,King of Thorns,Mark Lawrence,masculino,3.0,3.0,3.0
35105833,0.250,0.408,0.417,The Girl King,Mimi Yu,feminino,3.0,3.0,5.0


**Comparação entre coleções:** perfis médios de TF-IDF dos títulos de autoras e de autores, dentro de cada gênero literário.

In [10]:
# Comparar coleções: perfil médio de TF-IDF dos títulos de autoras e de autores, gênero a gênero.
# A matriz é construída separadamente em cada gênero literário (idf próprio de cada coleção).
def similaridade_entre_grupos(genero):
    d = e2[e2["genero_1"].eq(genero)].copy()
    d["tokens"] = d["titulo"].map(tokenizar_titulo)
    d = d[d["tokens"].map(len) > 0]
    m = pd.DataFrame([Counter(t) for t in d["tokens"]], index=d["id_livro"]).fillna(0)
    n = len(m)
    idf_g = ((n + 1) / (m.gt(0).sum() + 1)).map(math.log) + 1
    tfidf_g = m.mul(idf_g, axis=1)
    rotulo = d.set_index("id_livro")[GRUPO]
    medias = tfidf_g.groupby(rotulo).mean()
    vocab_f = set(m.columns[m[rotulo.eq("feminino").to_numpy()].sum() > 0])
    vocab_m = set(m.columns[m[rotulo.eq("masculino").to_numpy()].sum() > 0])
    return pd.Series({
        "titulos_fem": int(rotulo.eq("feminino").sum()), "titulos_masc": int(rotulo.eq("masculino").sum()),
        "cosseno_perfis_tfidf": cosseno(medias.loc["feminino"], medias.loc["masculino"]),
        "jaccard_vocabularios": len(vocab_f & vocab_m) / len(vocab_f | vocab_m),
    })

generos_comparados = ["Fantasy", "Romance", "Young Adult", "Fiction", "Historical", "Mystery", "Science Fiction", "Nonfiction"]
pd.DataFrame({g: similaridade_entre_grupos(g) for g in generos_comparados}).T.sort_values("cosseno_perfis_tfidf").round(3)

,titulos_fem,titulos_masc,cosseno_perfis_tfidf,jaccard_vocabularios
Science Fiction,126.0,300.0,0.150,0.060
Mystery,189.0,153.0,0.231,0.085
Historical,426.0,207.0,0.293,0.105
Young Adult,656.0,209.0,0.293,0.104
Romance,731.0,195.0,0.336,0.097
Fiction,432.0,754.0,0.373,0.121
Fantasy,675.0,483.0,0.398,0.145
Nonfiction,174.0,296.0,0.441,0.145


**Distância de edição** entre títulos quase idênticos de autores diferentes (as “versões” do recorte).

In [11]:
def levenshtein(seq_a, seq_b):
    anterior = list(range(len(seq_b) + 1))
    for i, a in enumerate(seq_a, 1):
        atual = [i]
        for j, b in enumerate(seq_b, 1):
            atual.append(min(anterior[j] + 1, atual[j - 1] + 1, anterior[j - 1] + (a != b)))
        anterior = atual
    return anterior[-1]

# "Versões": títulos com a mesma chave textual (Unidade 3) de autores diferentes
repetidos = base[base["titulo_repetido_outro_autor"]].sort_values(["titulo_chave", "ano_publicacao"])
linhas = []
for chave, grupo_titulos in repetidos.groupby("titulo_chave"):
    x, y = grupo_titulos.iloc[0], grupo_titulos.iloc[1]
    dist_c = levenshtein(x["titulo"], y["titulo"])
    tok_x, tok_y = x["titulo"].casefold().split(), y["titulo"].casefold().split()
    dist_t = levenshtein(tok_x, tok_y)
    linhas.append({"titulo A": x["titulo"], "autor A": x["autor"], "titulo B": y["titulo"], "autor B": y["autor"],
                   "dist_caracteres": dist_c, "dist_caracteres_norm": dist_c / max(len(x["titulo"]), len(y["titulo"])),
                   "dist_tokens": dist_t, "mesmo_autor": x["id_autor"] == y["id_autor"]})
pd.DataFrame(linhas).sort_values("dist_caracteres_norm").round(3)

,titulo A,autor A,titulo B,autor B,dist_caracteres,dist_caracteres_norm,dist_tokens,mesmo_autor
0,Angels Flight,Michael Connelly,Angels' Flight,Nalini Singh,1,0.071,1,False
4,In the Dark,Richard Laymon,In the dark,Jack Brighton,1,0.091,0,False
2,Fire Ice,Clive Cussler,Fire & Ice,Alice Brown,2,0.200,1,False
6,Leather+Lace,A.B. Gayle,Leather & Lace,Samantha A. Cole,3,0.214,3,False
8,"The Lion, the Witch and the Wardrobe",C.S. Lewis,"The Lion, the Witch and the Wardrobe (Chronicl...",Hiawyn Oram,27,0.429,4,False
7,The Last Summer (of You and Me),Ann Brashares,The Last Summer,Judith Kinghorn,16,0.516,4,False
5,Into the Fire,Ashelyn Drake,Into the Fire (Under Fire #1),Kim Vandel,16,0.552,3,False
3,Haunted (State v. Sefore #1),Charity Tinnin,Haunted,Kay Hooper,21,0.750,4,False
1,Blaze (or Love in the Time of Supervillains),Laurie Boyle Crompton,Blaze,Wolf Specter,39,0.886,7,False


A distância informa esforço mínimo de transformação, não gravidade editorial ou
mudança de sentido. Corrigir um nome próprio pode custar uma substituição e ser
substantivamente decisivo.

## 6. Comparar autores, períodos e coleções

Para grupos textuais, agregue somente após decidir se a unidade é documento,
autor ou coleção. Médias de vetores por período respondem a perfis médios; unir
todos os textos responde a um megadocumento e dá mais peso a grupos maiores.
Autoria não deve ser tratada como essência estilística: gênero, período,
transcrição e composição do corpus podem explicar parte da proximidade.

> **Nota sobre atividades em dupla, trio ou revisão por pares.** Esta resolução
> foi produzida por uma única pessoa. Onde o roteiro pede colegas, apresento uma
> **revisão crítica simulada**, guiada pelas perguntas do roteiro. Ela não
> substitui o trabalho e o parecer de colegas reais.

## U05-A05 — Atividade integrada — relatório de pares

**Modalidade:** trios. **Tempo:** 35 minutos.

1. escolha um documento-consulta e três candidatos;
2. calcule Jaccard e cosseno com duas ponderações;
3. identifique uma concordância e uma divergência entre rankings;
4. leia os documentos e metadados;
5. para versões, calcule distância de edição e descreva as mudanças;
6. justifique qual medida responde melhor à pergunta.

**Pares, métricas e resultados:** consulta = *The Girl of Fire and Thorns*
(Rae Carson, autoria feminina, Fantasy; tokens “girl”, “fire”, “thorns”).
Candidatos: *Into the Fire* (Kim Vandel e Ashelyn Drake, dois livros
diferentes), *King of Thorns* (Mark Lawrence) e *The Girl King* (Mimi Yu).

| Candidato | Jaccard | Cosseno (contagens) | Cosseno (TF-IDF) |
|---|---:|---:|---:|
| *Into the Fire* (Vandel; Drake) | 0,333 | 0,577 | 0,518 |
| *King of Thorns* (Lawrence) | 0,250 | 0,408 | 0,513 |
| *The Girl King* (Yu) | 0,250 | 0,408 | 0,417 |

*Concordância:* as três medidas colocam os dois *Into the Fire* em primeiro
lugar, porque compartilham “fire” e têm só dois termos de conteúdo.
*Divergência:* Jaccard e cosseno de contagens empatam *King of Thorns* e *The
Girl King* (um termo em comum cada), mas o TF-IDF separa os dois: “thorns” é
raro (df = 4) e pesa mais que “girl” (df = 11), então *King of Thorns* sobe
quase ao nível de *Into the Fire*, e *The Girl King* cai para a quinta posição.

**Leitura próxima e escolha da medida:** *King of Thorns*, de Mark Lawrence, é
fantasia sombria narrada por um príncipe; *The Girl of Fire and Thorns* é
fantasia de formação com uma protagonista. A proximidade por TF-IDF vem de uma
única palavra rara, não de afinidade temática. *Versões:* comparei os nove pares
de títulos que receberam a mesma chave textual na Unidade 3. “Angels Flight”
(Michael Connelly) e “Angels' Flight” (Nalini Singh) estão a 1 caractere de
distância; “In the Dark” e “In the dark”, também a 1 caractere (0 tokens depois
de minúsculas). *The Lion, the Witch and the Wardrobe* de C. S. Lewis e a versão
de Hiawyn Oram diferem em 27 caracteres, todos no marcador de série; trata-se
provavelmente de uma adaptação, não da mesma edição. Nenhum par é do mesmo
autor. *Escolha:* para a pergunta da pesquisa, que compara **coleções** (títulos
de autoras e de autores), a medida mais útil é o **cosseno entre perfis médios
de TF-IDF dentro de cada gênero literário** (tabela acima): ele mostra que a
proximidade entre os títulos dos dois grupos varia muito, de 0,15 em ficção
científica a 0,44 em não ficção, ou seja, o vocabulário dos títulos se separa
mais em ficção científica e mistério. Jaccard entre vocabulários vai no mesmo
sentido (0,06 em ficção científica). Ressalva: o cosseno de perfis médios
depende do número de títulos de cada grupo, porque médias de mais títulos são
mais suaves.

**O que a similaridade não demonstra:** que dois livros tratam do mesmo tema;
que há um “estilo feminino” ou “masculino” de dar títulos; que títulos quase
idênticos são a mesma obra (Connelly e Singh); ou que a proximidade vem do
gênero da autoria, e não de convenções de subgênero, de editoras ou de época.

Leve o relatório à oficina. A entrega final exigirá uma medida principal, uma
alternativa de sensibilidade e casos que alterem ou qualifiquem a conclusão.